# AttnRes — Three-way comparison on TinyStories

Trains and compares three model variants on the `roneneldan/TinyStories` dataset:

| # | Name | `use_block_attn_res` | `use_xsa` | Description |
|---|------|---------------------|-----------|-------------|
| 1 | **Baseline** | — | — | Standard fixed residuals (`h = h + attn + mlp`) |
| 2 | **AttnRes** | `true` | `false` | Block Attention Residuals |
| 3 | **AttnRes + XSA** | `true` | `true` | Block Attention Residuals + Exclusive Self Attention |

All runs are logged to **Weights & Biases** and checkpoints / logs are saved to **Google Drive**.

**Runtime:** GPU (T4 or better recommended). Set *Runtime → Change runtime type → GPU* before running.


## 0 · Runtime check

In [ ]:
import subprocess, sys

result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                         "--format=csv,noheader"], capture_output=True, text=True)
if result.returncode == 0:
    print("GPU:", result.stdout.strip())
else:
    print("⚠  No GPU detected — training will be slow on CPU.")
    print("   Go to Runtime → Change runtime type → GPU and re-run.")

import torch
print(f"PyTorch {torch.__version__}  |  CUDA available: {torch.cuda.is_available()}")


## 1 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
from pathlib import Path

# All output (checkpoints, logs, plots) goes here.
DRIVE_ROOT = Path("/content/drive/MyDrive/attnres_experiment")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

CKPT_ROOT = DRIVE_ROOT / "checkpoints"
LOG_ROOT  = DRIVE_ROOT / "logs"
PLOT_ROOT = DRIVE_ROOT / "plots"
for d in [CKPT_ROOT, LOG_ROOT, PLOT_ROOT]:
    d.mkdir(parents=True, exist_ok=True)

print("Drive root:", DRIVE_ROOT)


## 2 · Install dependencies

In [ ]:
# All installs are pip-based — no uv required.
%pip install -q wandb datasets transformers tokenizers tqdm rich
print("✓ Dependencies installed")


## 3 · Clone the AttnRes repository

In [ ]:
import os

REPO_DIR = Path("/content/attnres")

if not REPO_DIR.exists():
    !git clone https://github.com/MoonshotAI/Attention-Residuals.git /content/attnres
    print("✓ Cloned from GitHub")
else:
    print("✓ Repository already present — pulling latest changes")
    !git -C /content/attnres pull --quiet

# Make sure project root is on the Python path
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())


## 4 · Weights & Biases login

Paste your W&B API key when prompted, or set `WANDB_API_KEY` as a Colab secret
(*Key icon* in the left sidebar → Secrets) and run the cell below.


In [ ]:
import wandb

# Prefer Colab secrets (no interactive prompt needed in automated runs)
try:
    from google.colab import userdata
    api_key = userdata.get("WANDB_API_KEY")
    wandb.login(key=api_key, relogin=True)
    print("✓ Logged in via Colab secret")
except Exception:
    # Falls back to interactive prompt
    wandb.login()
    print("✓ Logged in interactively")


## 5 · Shared experiment configuration

Edit the values in this cell to change batch size, number of epochs, the
story cap for a quick smoke test, etc.  All three model variants share the
same hyper-parameters — only the architecture flags differ.


In [ ]:
# ── Experiment settings ────────────────────────────────────────────────────
WANDB_PROJECT    = "attnres-comparison"   # W&B project name
WANDB_ENTITY     = ""                     # W&B entity / team — leave "" for personal

# Number of training epochs (3 = full paper setup; reduce for a quick test)
EPOCHS           = 3

# Cap the training split to N stories.
# None = use all 2.12M stories (full training, several hours on a T4).
# 50_000 = ~20 min on T4, good enough for qualitative comparison.
MAX_TRAIN_STORIES = 50_000

# Model size (matches TinyStories-33M setup)
DIM        = 512
DEPTH      = 8
HEADS      = 8
HEAD_DIM   = 64

# Optimiser (matches TinyStories-33M paper)
BATCH_SIZE    = 32
LR            = 5e-4
WEIGHT_DECAY  = 0.1
WARMUP_STEPS  = 500
GRAD_CLIP     = 1.0
LOG_EVERY     = 200
SEQ_LEN       = 512

SEED = 42

print("Configuration:")
print(f"  MAX_TRAIN_STORIES : {MAX_TRAIN_STORIES:,}")
print(f"  EPOCHS            : {EPOCHS}")
print(f"  DIM/DEPTH/HEADS   : {DIM}/{DEPTH}/{HEADS}")
print(f"  BATCH_SIZE / LR   : {BATCH_SIZE} / {LR}")
print(f"  Drive root        : {DRIVE_ROOT}")


## 6 · Config builder

In [ ]:
from utils.config import (
    Config, ModelConfig, TrainingConfig, DataConfig,
    LoggingConfig, GenerationConfig,
)

def make_config(
    run_name: str,
    *,
    baseline: bool = False,
    use_block_attn_res: bool = True,
    use_xsa: bool = False,
) -> Config:
    """Return a fully-populated Config for one experiment variant."""
    ckpt_dir = str(CKPT_ROOT / run_name)
    log_dir  = str(LOG_ROOT  / run_name)

    model = ModelConfig(
        name               = run_name,
        dim                = DIM,
        depth              = DEPTH,
        heads              = HEADS,
        head_dim           = HEAD_DIM,
        mlp_multiplier     = 4,
        dropout            = 0.1,
        use_block_attn_res = use_block_attn_res,
        block_size         = 4,
        norm_eps           = 1e-6,
        max_seq_len        = SEQ_LEN,
        use_kv_cache       = False,    # inference-only; never True during training
        use_xsa            = use_xsa,
    )
    training = TrainingConfig(
        epochs       = EPOCHS,
        batch_size   = BATCH_SIZE,
        lr           = LR,
        weight_decay = WEIGHT_DECAY,
        grad_clip    = GRAD_CLIP,
        warmup_steps = WARMUP_STEPS,
        log_every    = LOG_EVERY,
        save_every   = 1,
        seed         = SEED,
        device       = "auto",
    )
    data = DataConfig(
        dataset     = "tinystories",
        data_dir    = str(DRIVE_ROOT / "data"),
        seq_len     = SEQ_LEN,
        stride      = SEQ_LEN,       # non-overlapping windows
        num_workers = 2,
        pin_memory  = True,
        val_split   = 0.0,
    )
    logging = LoggingConfig(
        log_dir              = log_dir,
        checkpoint_dir       = ckpt_dir,
        tracker              = "wandb",
        wandb_project        = WANDB_PROJECT,
        wandb_run_name       = run_name,
        mlflow_tracking_uri  = "mlruns",
        mlflow_experiment    = "AttnRes",
    )
    generation = GenerationConfig(
        max_new_tokens = 200,
        temperature    = 0.8,
        top_k          = 40,
    )
    return Config(
        model=model, training=training, data=data,
        logging=logging, generation=generation,
    )

# ── Define the three variants ─────────────────────────────────────────────
VARIANTS = [
    dict(run_name="01_Baseline",      baseline=True,  use_block_attn_res=False, use_xsa=False),
    dict(run_name="02_AttnRes",       baseline=False, use_block_attn_res=True,  use_xsa=False),
    dict(run_name="03_AttnRes_XSA",   baseline=False, use_block_attn_res=True,  use_xsa=True),
]

for v in VARIANTS:
    cfg = make_config(**v)
    print(f"{v['run_name']:25s}  baseline={v['baseline']}  "
          f"block_attn_res={v['use_block_attn_res']}  xsa={v['use_xsa']}  "
          f"ckpt → {cfg.logging.checkpoint_dir}")


## 7 · Train all three variants

Each variant is trained sequentially in the same cell.  The tokenised dataset
is downloaded and cached once on first run; subsequent variants reuse the same
cached tensors from Drive.

W&B runs appear in the project **`attnres-comparison`** as they start.


In [ ]:
import sys, os, gc
import torch
sys.path.insert(0, str(REPO_DIR))

from train.train_lm import train_lm

RESULTS = {}   # {run_name: {"val_loss": float, "val_ppl": float}}

for variant in VARIANTS:
    run_name = variant["run_name"]
    print("\n" + "═" * 60)
    print(f"  Training: {run_name}")
    print("═" * 60)

    cfg = make_config(**variant)

    # Create output dirs on Drive
    Path(cfg.logging.checkpoint_dir).mkdir(parents=True, exist_ok=True)
    Path(cfg.logging.log_dir).mkdir(parents=True, exist_ok=True)

    try:
        train_lm(
            cfg,
            baseline          = variant["baseline"],
            resume            = False,
            max_train_stories = MAX_TRAIN_STORIES,
        )
        print(f"✓ {run_name} complete")

        # Read the best checkpoint val_loss for the summary table
        best_ckpt = Path(cfg.logging.checkpoint_dir) / "best" / "best.pt"
        if best_ckpt.exists():
            ckpt_data = torch.load(best_ckpt, map_location="cpu", weights_only=False)
            vl = ckpt_data.get("val_loss", float("nan"))
            import math
            RESULTS[run_name] = {"val_loss": vl, "val_ppl": math.exp(min(vl, 20))}

    except Exception as exc:
        print(f"✗ {run_name} FAILED: {exc}")
        import traceback; traceback.print_exc()

    # Free GPU memory before the next run
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n✓ All training runs finished")


## 8 · Results summary

In [ ]:
import math
import pandas as pd

rows = []
for variant in VARIANTS:
    run_name = variant["run_name"]
    res = RESULTS.get(run_name, {})
    rows.append({
        "Run":                run_name,
        "Baseline":           "✓" if variant["baseline"] else "—",
        "Block AttnRes":      "✓" if variant["use_block_attn_res"] else "—",
        "XSA":                "✓" if variant["use_xsa"] else "—",
        "Best val loss":      f"{res.get('val_loss', float('nan')):.4f}",
        "Best val ppl":       f"{res.get('val_ppl', float('nan')):.2f}",
    })

df = pd.DataFrame(rows).set_index("Run")
print(df.to_string())


## 9 · Training curve comparison

Reads the CSV logs saved to Drive and plots validation loss and perplexity
for all three variants on the same axes.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path

COLORS = {
    "01_Baseline":    "#ef4444",   # red
    "02_AttnRes":     "#6366f1",   # indigo
    "03_AttnRes_XSA": "#10b981",   # emerald
}
LABELS = {
    "01_Baseline":    "Baseline (fixed residuals)",
    "02_AttnRes":     "AttnRes (Block, no XSA)",
    "03_AttnRes_XSA": "AttnRes + XSA",
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ax_loss, ax_ppl, ax_tps = axes

plot_ok = False
for variant in VARIANTS:
    run_name = variant["run_name"]
    log_dir  = LOG_ROOT / run_name
    csv_files = sorted(log_dir.glob("*.csv")) if log_dir.exists() else []
    if not csv_files:
        print(f"No CSV found for {run_name} — skipping plot")
        continue

    df = pd.read_csv(csv_files[-1])              # most recent run
    epoch_df = df[df["phase"] == "epoch"].copy()
    if epoch_df.empty:
        print(f"No epoch rows in {csv_files[-1].name} — skipping plot")
        continue

    epochs   = epoch_df["epoch"].astype(int)
    val_loss = epoch_df["val_loss"].astype(float)
    val_ppl  = (val_loss.apply(lambda x: min(x, 20))).apply(lambda x: 2.718281828**x)
    color    = COLORS.get(run_name, "#888888")
    label    = LABELS.get(run_name, run_name)

    ax_loss.plot(epochs, val_loss, marker="o", color=color, label=label, linewidth=2)
    ax_ppl.plot( epochs, val_ppl,  marker="o", color=color, label=label, linewidth=2)

    # Throughput (tokens/sec) — step-phase rows only
    step_df = df[(df["phase"] == "train") & (df["tokens_per_sec"] != "")].copy()
    if not step_df.empty:
        step_df["tokens_per_sec"] = step_df["tokens_per_sec"].astype(float)
        ax_tps.plot(
            step_df["step"].astype(int),
            step_df["tokens_per_sec"] / 1000,
            color=color, label=label, linewidth=1.2, alpha=0.8,
        )
    plot_ok = True

if plot_ok:
    for ax, ylabel, title in [
        (ax_loss, "Validation loss",       "Val loss per epoch"),
        (ax_ppl,  "Validation perplexity", "Val perplexity per epoch"),
        (ax_tps,  "Throughput (k tok/s)",  "Training throughput"),
    ]:
        ax.set_xlabel("Epoch" if ax != ax_tps else "Step")
        ax.set_ylabel(ylabel)
        ax.set_title(title, fontweight="bold")
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

    plt.suptitle("AttnRes three-way comparison — TinyStories", fontsize=14, fontweight="bold")
    plt.tight_layout()

    plot_path = PLOT_ROOT / "comparison.png"
    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    print(f"Plot saved → {plot_path}")
    plt.show()
else:
    print("No CSV data available to plot yet.")


## 10 · Generate text samples from each model

Loads the best checkpoint for each variant and generates a short story
continuation from the same seed prompt, with and without KV cache.


In [ ]:
import torch, math, time
from pathlib import Path
from models.lm_transformer import AttnResLM, BaselineLM
from utils.config import ModelConfig

PROMPT = "Once upon a time there was a little girl named Lily who"
MAX_NEW = 150
TEMPERATURE = 0.8
TOP_K = 40

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the tokeniser once (shared across all variants)
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-125M")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

prompt_ids = torch.tensor(
    tokenizer.encode(PROMPT), dtype=torch.long, device=device
).unsqueeze(0)

for variant in VARIANTS:
    run_name  = variant["run_name"]
    best_ckpt = CKPT_ROOT / run_name / "best" / "best.pt"
    print("\n" + "─" * 60)
    print(f"  {run_name}")
    print("─" * 60)

    if not best_ckpt.exists():
        print(f"  ✗ No checkpoint found at {best_ckpt}")
        continue

    ckpt_data = torch.load(best_ckpt, map_location=device, weights_only=False)
    raw_cfg   = ckpt_data.get("config", {}).get("model", {})
    model_cfg = ModelConfig(**raw_cfg)
    vocab_size = tokenizer.vocab_size

    ModelClass = BaselineLM if variant["baseline"] else AttnResLM
    model = ModelClass(model_cfg, vocab_size=vocab_size, seq_len=model_cfg.max_seq_len)
    model.load_state_dict(ckpt_data["model_state"])
    model.eval().to(device)

    val_loss = ckpt_data.get("val_loss", float("nan"))
    print(f"  Checkpoint val loss: {val_loss:.4f}  ppl: {math.exp(min(val_loss,20)):.2f}")

    # Generate without KV cache
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(prompt_ids, max_new_tokens=MAX_NEW,
                             temperature=TEMPERATURE, top_k=TOP_K,
                             use_kv_cache=False)
    t_no_cache = time.perf_counter() - t0
    text_no_cache = tokenizer.decode(out[0].tolist())

    # Generate with KV cache
    t0 = time.perf_counter()
    with torch.no_grad():
        out_cached = model.generate(prompt_ids, max_new_tokens=MAX_NEW,
                                    temperature=TEMPERATURE, top_k=TOP_K,
                                    use_kv_cache=True)
    t_cached = time.perf_counter() - t0
    text_cached = tokenizer.decode(out_cached[0].tolist())

    new_toks = MAX_NEW
    print(f"  No cache: {t_no_cache:.2f}s  {new_toks/t_no_cache:.1f} tok/s")
    print(f"  KV cache: {t_cached:.2f}s  {new_toks/t_cached:.1f} tok/s  "
          f"({t_no_cache/t_cached:.1f}× speedup)")
    print("\nGenerated text:")
    print(text_no_cache[:500])

    # Free model memory
    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None


## 11 · Inference speed comparison

In [ ]:
import pandas as pd, math, time, torch, gc
from pathlib import Path
from models.lm_transformer import AttnResLM, BaselineLM
from utils.config import ModelConfig
from transformers import AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-125M")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

prompt_ids = torch.tensor(
    tokenizer.encode("Once upon a time"), dtype=torch.long, device=device
).unsqueeze(0)

N_TOKENS = 200
N_REPEATS = 3   # average over N_REPEATS runs for stable timing

rows = []
for variant in VARIANTS:
    run_name  = variant["run_name"]
    best_ckpt = CKPT_ROOT / run_name / "best" / "best.pt"
    if not best_ckpt.exists():
        rows.append({"Run": run_name, "Val loss": "—", "Val PPL": "—",
                     "tok/s (no cache)": "—", "tok/s (KV cache)": "—",
                     "Cache speedup": "—"})
        continue

    ckpt_data = torch.load(best_ckpt, map_location=device, weights_only=False)
    raw_cfg   = ckpt_data.get("config", {}).get("model", {})
    model_cfg = ModelConfig(**raw_cfg)
    ModelClass = BaselineLM if variant["baseline"] else AttnResLM
    model = ModelClass(model_cfg, vocab_size=tokenizer.vocab_size,
                       seq_len=model_cfg.max_seq_len)
    model.load_state_dict(ckpt_data["model_state"])
    model.eval().to(device)

    vl = ckpt_data.get("val_loss", float("nan"))

    def bench(use_cache):
        times = []
        for _ in range(N_REPEATS):
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            with torch.no_grad():
                model.generate(prompt_ids, max_new_tokens=N_TOKENS,
                               temperature=1.0, top_k=None, use_kv_cache=use_cache)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            times.append(time.perf_counter() - t0)
        return N_TOKENS / (sum(times) / len(times))

    tps_no  = bench(use_cache=False)
    tps_yes = bench(use_cache=True)

    rows.append({
        "Run":              run_name,
        "Val loss":         f"{vl:.4f}",
        "Val PPL":          f"{math.exp(min(vl,20)):.2f}",
        "tok/s (no cache)": f"{tps_no:.1f}",
        "tok/s (KV cache)": f"{tps_yes:.1f}",
        "Cache speedup":    f"{tps_yes/tps_no:.2f}×",
    })

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

df_speed = pd.DataFrame(rows).set_index("Run")
print(df_speed.to_string())

# Save table to Drive as CSV
speed_path = PLOT_ROOT / "inference_speed.csv"
df_speed.to_csv(speed_path)
print(f"\nTable saved → {speed_path}")


## 12 · Verify Drive contents

In [ ]:
import os
from pathlib import Path

def tree(root, max_depth=3, prefix=""):
    root = Path(root)
    if not root.exists():
        print(f"{prefix}{root.name}  (not found)")
        return
    entries = sorted(root.iterdir())
    for i, entry in enumerate(entries):
        connector = "└── " if i == len(entries) - 1 else "├── "
        size = ""
        if entry.is_file():
            sz = entry.stat().st_size
            size = f"  ({sz/1024:.0f} KB)" if sz > 1024 else f"  ({sz} B)"
        print(f"{prefix}{connector}{entry.name}{size}")
        if entry.is_dir() and max_depth > 1:
            ext = "    " if i == len(entries) - 1 else "│   "
            tree(entry, max_depth - 1, prefix + ext)

print("Google Drive — AttnRes experiment folder")
print("=" * 50)
tree(DRIVE_ROOT)
